In [1]:
from pathlib import Path
import sys

project_root = Path.cwd()
if (project_root / "src").is_dir():
    pass
elif (project_root / "cricket-win-predict" / "src").is_dir():
    project_root = project_root / "cricket-win-predict"
elif project_root.name == "notebooks" and (project_root.parent / "src").is_dir():
    project_root = project_root.parent
else:
    raise FileNotFoundError("Could not locate the cricket-win-predict project root")

sys.path.insert(0, str(project_root))

In [ ]:
from src.data.loader import load_data, filter_with_nation_winners
from src.data.prepare_data import DatasetLoad

In [ ]:
data = load_data()
data = filter_with_nation_winners(data)

dataset = DatasetLoad(data, to_insert_team_indices=True)
_, second_innings_data = dataset.process_data()
len(second_innings_data)

2040

In [55]:
import pandas as pd
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(second_innings_data, test_size=0.2)

train_data = [state for match in train_data for state in match]
test_data = [state for match in test_data for state in match]


train_df = pd.DataFrame(train_data)
test_df = pd.DataFrame(test_data)

In [56]:
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer


preprocessor = ColumnTransformer(
    transformers=[
        ("cat", 
         OneHotEncoder(handle_unknown="ignore", sparse_output=False).set_output(transform="pandas"), 
         ["batting_team", "bowling_team"]),
        # ("num", MinMaxScaler(), ["runs", "wickets", "balls"])
    ],
    remainder="passthrough"
)

# train_df["runs"] = train_df["runs"] / 300
# train_df["balls"] = train_df["balls"] / 300
# train_df["wickets"] = train_df["wickets"] / 10

# test_df["runs"] = test_df["runs"] / 300
# test_df["balls"] = test_df["balls"] / 300
# test_df["wickets"] = test_df["wickets"] / 10
X_train, y_train = train_df.drop(columns=["labels"]), train_df["labels"]
X_test, y_test = test_df.drop(columns=["labels"]), test_df["labels"]

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)


In [57]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

model = LogisticRegression(max_iter=400)
model.fit(X_train, y_train)
log_loss(y_test, model.predict_proba(X_test)), log_loss(y_train, model.predict_proba(X_train))


/home/rohan/TimePass/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 400 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=400).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


(0.39037190955628465, 0.3522218108219369)